In [2]:
import os
import json
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")

C:\Users\Nittan Kumar\AppData\Local\Temp\ipykernel_6864\2337123905.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [3]:
SYSTEM_PROMPT = """
You are a helpful assistant that answers questions related to software development and programming. You have access to the following tools:
1. Code Search: Search for code snippets and examples related to programming questions.
2. Documentation Search: Search for official documentation and resources related to programming languages, libraries, and frameworks.
3. Stack Overflow Search: Search for relevant questions and answers on Stack Overflow.
When a user asks a question, you should first determine which tool(s) to use to find the most relevant information. You can use multiple tools if necessary. After gathering information from the tools, you should provide a comprehensive and accurate answer to the user's question.
Remember to always cite your sources and provide links to the information you found. If you are unsure about an answer, it's better to say "I don't know" rather than providing incorrect information."""

In [4]:
HISTORY_FILE = "conversation/history.json"

def load_history():
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE, "r") as f:
            return json.load(f)
    return []

def save_history(history):
    with open(HISTORY_FILE, "w") as f:
        json.dump(history, f, indent=4)

In [5]:
def ask_ai(question):
    history = load_history()

    history.append({"role": "user", "content": question})
    conversation = ""

    for msg in history:
        conversation += f"{msg['role']}: {msg['content']}\n"
    final_prompt = SYSTEM_PROMPT + "\n" + conversation + "\n Assistant:" + "\n"

    response = model.generate_content(final_prompt)
    answer = response.text
    history.append({"role": "assistant", "content": answer})
    save_history(history)
    return answer

In [6]:
while True:
    question = input("Ask a question (or type 'exit' to quit): ")
    if question.lower() == "exit":
        break
    answer = ask_ai(question)
    print("Answer:", answer)

Similarly we can use hugging face model as well to run on our computer even without internet. 

In [7]:
! pip install --upgrade --quiet \
    langchain-huggingface \
    text-generation \
    transformers \
    langchainhub \
    bitsandbytes \
    accelerate

In [8]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
import os
import getpass

In [9]:
os.environ["HUGGINGFACE_API_KEY"] = getpass.getpass()

In [10]:
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type= "nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant= True
)

In [11]:
import requests

response = requests.get("https://huggingface.co")
print(response.status_code)  # Should print 200 if the request was successful

200


Requires lot of memory

In [ ]:

from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

llm = HuggingFacePipeline.from_model_id(
    # model_id="google/gemma-2b",
    model_id="microsoft/Phi-3-mini-4k-instruct",
    task="text-generation",
    pipeline_kwargs=dict(
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.03,
        return_full_text=False,
    ),
    model_kwargs={"quantization_config": quantization_config},
)

chat_model = ChatHuggingFace(llm=llm)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]c:\Users\Nittan Kumar\Desktop\Documents\Learn Gen AI\minor projects\venv\Lib\site-packages\bitsandbytes\backends\default\ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\Nittan Kumar\Desktop\Documents\Learn Gen AI\minor projects\venv\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Now we will work with ollama by downloading it. 

eg. ollama run phi3

In [1]:
pip install langchain-community

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   ---------------------- ----------------- 1.3/2.4 MB 3.8 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 4.2 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 5.4 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ------------------------ --------------- 1.3/2.1 MB 6.1 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 6.1 MB/s  0:00:00

   ---------------------------------------- 0/7 [httpx-sse]
   ----- ---------------------------------- 1/7 [greenlet]
   ----- ---------------------------------- 1/7 [greenlet]
   ----- ---------------------------------- 1/7 [greenlet]
   ----- ---------------------------------- 1/

In [ ]:
!pip install langchain-ollama
from langchain_ollama import ChatOllama

chat_model = ChatOllama(
    model = "phi3",
    temperature = 0.1,
    max_tokens = 512)

response = chat_model.invoke("What is the capital of France?")
print(response.content)

ImportError: cannot import name 'ChatOllama' from 'langchain_community.chat_models' (c:\Users\Nittan Kumar\Desktop\Documents\Learn Gen AI\minor projects\venv\Lib\site-packages\langchain_community\chat_models\__init__.py)